# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kareem5545163-art/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Install the datasets library if needed (already available in most Colab sessions)
!pip install -q datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

# Load the FlyRank starter dataset directly from Hugging Face
ds = load_dataset("FlyRank/internship-starter", split="train")
df = ds.to_pandas()

declining = df["trend_direction"] == "down"
print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()}")
print(f"declining rate overall: {declining.mean():.3f}  ({declining.sum():,} of {len(df):,})")

stale = df["freshness_tier"].isin(["91-180", "181+"])
print(f"\nstale pages (freshness_tier 91-180/181+): n={stale.sum():,}")
print(f"declining rate | stale:     {df.loc[stale, 'trend_direction'].eq('down').mean():.3f}")
print(f"declining rate | not stale: {df.loc[~stale, 'trend_direction'].eq('down').mean():.3f}")

med_imp_decl   = df.loc[declining, "impressions_90d"].median()
med_imp_stable = df.loc[~declining, "impressions_90d"].median()
print(f"\nmedian impressions_90d | declining:     {med_imp_decl:.0f}")
print(f"median impressions_90d | not declining: {med_imp_stable:.0f}")
print(f"ratio: {med_imp_decl / med_imp_stable:.2f}x")

README.md:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

content_refresh_anonymized.csv:   0%|          | 0.00/8.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30000 [00:00<?, ? examples/s]

rows: 30,000 | clients: 32
declining rate overall: 0.542  (16,262 of 30,000)

stale pages (freshness_tier 91-180/181+): n=9,345
declining rate | stale:     0.608
declining rate | not stale: 0.512

median impressions_90d | declining:     961
median impressions_90d | not declining: 472
ratio: 2.04x


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = df["impression_tier"].isin(["good", "excellent"])
stale_and_visible = stale & visible

print(f"declining rate | stale AND visible: {df.loc[stale_and_visible, 'trend_direction'].eq('down').mean():.3f}  (n={stale_and_visible.sum():,})")
print(f"declining rate | stale alone:       {df.loc[stale, 'trend_direction'].eq('down').mean():.3f}  (n={stale.sum():,})")
print("-> stacking a visibility filter on top of staleness barely beats staleness alone:")
print("   staleness is doing almost all the work, and neither rule cleanly separates decline from non-decline.")

pool_declining = declining.sum()
pool_declining_visible = (declining & visible).sum()
print(f"\nreview-pool scale: {pool_declining:,} pages are declining;")
print(f"{pool_declining_visible:,} of those are also visible enough to matter (impression_tier good/excellent).")
print("Either pool is far bigger than an editor can review by hand in a normal cycle -> ranking, not just filtering, is needed.")

declining rate | stale AND visible: 0.576  (n=3,524)
declining rate | stale alone:       0.608  (n=9,345)
-> stacking a visibility filter on top of staleness barely beats staleness alone:
   staleness is doing almost all the work, and neither rule cleanly separates decline from non-decline.

review-pool scale: 16,262 pages are declining;
4,721 of those are also visible enough to matter (impression_tier good/excellent).
Either pool is far bigger than an editor can review by hand in a normal cycle -> ranking, not just filtering, is needed.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"1) {len(df):,} pages across {df['client_id'].nunique()} clients; "
      f"{declining.mean():.1%} ({declining.sum():,}) are currently declining (trend_direction == 'down').")
print(f"2) Stale pages (not updated in 91+ days, n={stale.sum():,}) decline at "
      f"{df.loc[stale, 'trend_direction'].eq('down').mean():.1%} vs {declining.mean():.1%} overall "
      f"(+{(df.loc[stale, 'trend_direction'].eq('down').mean() - declining.mean())*100:.1f} points).")
print(f"3) Declining pages carry ~{med_imp_decl/med_imp_stable:.1f}x the traffic of non-declining pages "
      f"(median impressions_90d {med_imp_decl:.0f} vs {med_imp_stable:.0f}).")

1) 30,000 pages across 32 clients; 54.2% (16,262) are currently declining (trend_direction == 'down').
2) Stale pages (not updated in 91+ days, n=9,345) decline at 60.8% vs 54.2% overall (+6.6 points).
3) Declining pages carry ~2.0x the traffic of non-declining pages (median impressions_90d 961 vs 472).


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
by_client = df.groupby("client_id")["trend_direction"].apply(lambda s: (s == "down").mean())
n_by_client = df.groupby("client_id").size()

print(f"declining rate across the {by_client.shape[0]} clients ranges from {by_client.min():.1%} to {by_client.max():.1%}")
print(f"rows per client range from {n_by_client.min()} to {n_by_client.max():,} -> a very unbalanced panel")

print("\ntrend_direction fully determines the down/not-down split by construction;")
print("trend_direction and trend_pct will be excluded from any future feature set.")

declining rate across the 32 clients ranges from 0.0% to 93.7%
rows per client range from 3 to 7,008 -> a very unbalanced panel

trend_direction fully determines the down/not-down split by construction;
trend_direction and trend_pct will be excluded from any future feature set.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.